In [1]:
import argparse
import numpy as np
import torch
from tqdm import tqdm
from eval_jux import Solver as eval_jux_solver
from sklearn.neighbors import KernelDensity

from util import ECFL


# set the random seed manually for reproducibility
SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [2]:
parser = argparse.ArgumentParser()


parser.add_argument('--device', default='cpu', type=str,
                    help='cpu/cuda')
parser.add_argument('--batch_size', default=1, type=int,
                    help='batch size')
parser.add_argument('--max_iter', default=0, type=float,
                    help='maximum number of batch iterations')
parser.add_argument('--ckpt_dir', default='ckpts', type=str)
parser.add_argument('--scale', default=1.0, type=float)
parser.add_argument('--num_goal', default=3, type=int)
parser.add_argument('--n_z', default=1, type=int)

# nuScenes Params
parser.add_argument('--dt', default=0.5, type=float)
parser.add_argument('--obs_len', default=4, type=int)
parser.add_argument('--pred_len', default=12, type=int)
parser.add_argument('--dataset_dir', default='', type=str, help='dataset directory')
parser.add_argument('--dataset_name', default='nuScenes', type=str,
                    help='dataset name')
parser.add_argument('--heatmap_size', default=256, type=int)
parser.add_argument('--diffusion_micro_cfg_id', default='nuscenes_micro_diffuse', type=str)
parser.add_argument('--n_w', default=5, type=int)


args = parser.parse_args(args=[])

In [3]:
# nuScenes
def nuscenes_ECFL(solver, output):
    curr = 0
    ECFL_total = 0
    pbar = tqdm(total=len(solver.data_loader.idx_list))
    while not solver.data_loader.is_epoch_end():
        pbar.update(1)
        batch = solver.data_loader.next_sample()
        # print('compute {} out of {}'.format(solver.data_loader.index, len(solver.data_loader.idx_list)))
        if batch is None:
            continue
        (obs_traj, fut_traj, obs_traj_st, fut_vel_st, seq_start_end,
            map_info, inv_h_t,
            local_map, local_ic, local_homo) = batch
        global_traj = output[:,curr:curr+seq_start_end[0][-1],:,:].transpose(0,1)
        curr +=seq_start_end[0][-1]
        ECFL_total += ECFL(global_traj, map_info[0])
    pbar.close()
    return ECFL_total/(output.shape[0]*output.shape[1])

In [5]:
def DE_by_timestep(outputs, gt_futures):
    distance_to_gt = ((outputs-gt_futures)**2).sum(-1).sqrt()
    sorted_ade_mode_idx = distance_to_gt.mean(-1).sort(0).indices
    sorted_fde_mode_idx = distance_to_gt[...,-1].sort(0).indices
    cum_ade = []
    cum_fde = []
    for i in range(distance_to_gt.shape[-1]):
        cum_ade.append(torch.gather(distance_to_gt[...,:i+1].mean(-1), dim=0, index=sorted_ade_mode_idx)[0].mean().item())
        cum_fde.append(torch.gather(distance_to_gt[...,i], dim=0, index=sorted_fde_mode_idx)[0].mean().item())
    
    return cum_ade, cum_fde

In [6]:
def kde_nll(outputs, gt_futures):
    log_dens = 0.0
    for i in tqdm(range(len(gt_futures))):
        X = outputs[:,i,:,:].cpu().numpy()
        Y = gt_futures[i,:,:].cpu().numpy()
        for t in range(len(Y)):
            kde = KernelDensity(kernel='gaussian', bandwidth=1).fit(X[:,t,:])
            log_dens += kde.score_samples(Y[t].reshape(1, -1)).item()

    ll = log_dens/gt_futures.shape[0]/gt_futures.shape[1]
    return -1*ll


In [ ]:
# eval script for dataloader
solver = eval_jux_solver(args)


-------------------------- loading test data --------------------------
0 sequence files are loaded ...
total num samples: 0
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


In [ ]:
# Trajectron++ Nuscenes 5
solver = eval_jux_solver(args)
trajectron_nuscenes_5 = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_trajpp.pt')
trajectron_nuscenes_5_gt = trajectron_nuscenes_5['gt']
trajectron_nuscenes_5_output = trajectron_nuscenes_5['sample']
trajectron_nuscenes_5_cum_ade, trajectron_nuscenes_5_cum_fde = DE_by_timestep(trajectron_nuscenes_5_output, trajectron_nuscenes_5_gt)
trajectron_nuscenes_5_kde_nll = kde_nll(trajectron_nuscenes_5_output, trajectron_nuscenes_5_gt)
trajectron_nuscenes_5_ECFL = nuscenes_ECFL(solver, trajectron_nuscenes_5_output)
print("=====")
print("Trajectron++ Nuscenes 5")
print("ADE: ", trajectron_nuscenes_5_cum_ade[-1])
print("FDE: ", trajectron_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", trajectron_nuscenes_5_kde_nll)
print("ECFL: ", trajectron_nuscenes_5_ECFL)
print("=====")


-------------------------- loading test data --------------------------


/tmp/ipykernel_3590186/3101085360.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trajectron_nuscenes_5 = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_trajpp

138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [02:16<00:00, 25.32it/s]

=====
Trajectron++ Nuscenes 5
ADE:  2.514879563135955
FDE:  5.571340176010946
KDE NLL:  11.655575269678934
ECFL:  0.8166353279504479
=====


In [16]:
# Agentformer Nuscenes 5
solver = eval_jux_solver(args)
agentformer_nuscenes_5 = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_aformer.pt')
agentformer_nuscenes_5_gt = agentformer_nuscenes_5['gt']
agentformer_nuscenes_5_output = agentformer_nuscenes_5['sample']
agentformer_nuscenes_5_cum_ade, agentformer_nuscenes_5_cum_fde = DE_by_timestep(agentformer_nuscenes_5_output, agentformer_nuscenes_5_gt)
agentformer_nuscenes_5_kde_nll = kde_nll(agentformer_nuscenes_5_output, agentformer_nuscenes_5_gt)
agentformer_nuscenes_5_ECFL = nuscenes_ECFL(solver, agentformer_nuscenes_5_output)
print("=====")
print("Agentformer Nuscenes 5")
print("ADE: ", agentformer_nuscenes_5_cum_ade[-1])
print("FDE: ", agentformer_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", agentformer_nuscenes_5_kde_nll)
print("ECFL: ", agentformer_nuscenes_5_ECFL)
print("=====")


-------------------------- loading test data --------------------------
138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [02:43<00:00, 21.13it/s]

=====
Agentformer Nuscenes 5
ADE:  1.8557982444763184
FDE:  3.889230728149414
KDE NLL:  6.946923596416013
ECFL:  0.8466320097334366
=====


In [17]:
#Ynet Nuscenes 5
solver = eval_jux_solver(args)
ynet_nuscenes_5 = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_ynet.pt')
ynet_nuscenes_5_gt = ynet_nuscenes_5['gt']
ynet_nuscenes_5_output = ynet_nuscenes_5['sample']
ynet_nuscenes_5_cum_ade, ynet_nuscenes_5_cum_fde = DE_by_timestep(ynet_nuscenes_5_output, ynet_nuscenes_5_gt)
ynet_nuscenes_5_kde_nll = kde_nll(ynet_nuscenes_5_output, ynet_nuscenes_5_gt)
ynet_nuscenes_5_ECFL = nuscenes_ECFL(solver, ynet_nuscenes_5_output)
print("=====")
print("Ynet Nuscenes 5")
print("ADE: ", ynet_nuscenes_5_cum_ade[-1])
print("FDE: ", ynet_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", ynet_nuscenes_5_kde_nll)
print("ECFL: ", ynet_nuscenes_5_ECFL)
print("=====")


-------------------------- loading test data --------------------------
138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [02:42<00:00, 21.27it/s]

=====
Ynet Nuscenes 5
ADE:  1.629638433456421
FDE:  2.863093852996826
KDE NLL:  7.134641798256575
ECFL:  0.7660878221435682
=====


In [18]:
muse_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_muse.pt')
muse_nuscenes_5_gt = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_gt_futures.pt')
muse_nuscenes_5_cum_ade, muse_nuscenes_5_cum_fde = DE_by_timestep(muse_nuscenes_5_output, muse_nuscenes_5_gt)
muse_nuscenes_5_kde_nll = kde_nll(muse_nuscenes_5_output, muse_nuscenes_5_gt)
muse_nuscenes_5_ECFL = nuscenes_ECFL(solver, muse_nuscenes_5_output)
print("=====")
print("Muse Nuscenes 5")
print("ADE: ", muse_nuscenes_5_cum_ade[-1])
print("FDE: ", muse_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", muse_nuscenes_5_kde_nll)
print("ECFL: ", muse_nuscenes_5_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [02:44<00:00, 20.99it/s]

=====
Muse Nuscenes 5
ADE:  1.3681179285049438
FDE:  2.840756893157959
KDE NLL:  5.761968560316891
ECFL:  0.8929543192124765
=====


In [16]:
diffuse_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_diffuse.pt')
diffuse_nuscenes_5_gt = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_gt_futures.pt')
diffuse_nuscenes_5_cum_ade, diffuse_nuscenes_5_cum_fde = DE_by_timestep(diffuse_nuscenes_5_output, diffuse_nuscenes_5_gt)
diffuse_nuscenes_5_kde_nll = kde_nll(diffuse_nuscenes_5_output, diffuse_nuscenes_5_gt)
diffuse_nuscenes_5_ECFL = nuscenes_ECFL(solver, diffuse_nuscenes_5_output)
print("=====")
print("Diffuse Nuscenes 5")
print("ADE: ", diffuse_nuscenes_5_cum_ade[-1])
print("FDE: ", diffuse_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", diffuse_nuscenes_5_kde_nll)
print("ECFL: ", diffuse_nuscenes_5_ECFL)
print("=====")


/tmp/ipykernel_3590186/2048283483.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  diffuse_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_di

=====
Diffuse Nuscenes 5
ADE:  1.6815162897109985
FDE:  2.7550852298736572
KDE NLL:  6.938482394679503
ECFL:  0.9209158278951444
=====


In [17]:
guided_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_guided.pt')
guided_nuscenes_5_gt = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_gt_futures.pt')
guided_nuscenes_5_cum_ade, guided_nuscenes_5_cum_fde = DE_by_timestep(guided_nuscenes_5_output, guided_nuscenes_5_gt)
guided_nuscenes_5_kde_nll = kde_nll(guided_nuscenes_5_output, guided_nuscenes_5_gt)
guided_nuscenes_5_ECFL = nuscenes_ECFL(solver, guided_nuscenes_5_output)
print("=====")
print("Guided Nuscenes 5")
print("ADE: ", guided_nuscenes_5_cum_ade[-1])
print("FDE: ", guided_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", guided_nuscenes_5_kde_nll)
print("ECFL: ", guided_nuscenes_5_ECFL)
print("=====")


/tmp/ipykernel_3590186/1402311722.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  guided_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_gui

=====
Guided Nuscenes 5
ADE:  1.669740915298462
FDE:  2.7289600372314453
KDE NLL:  6.853946574622252
ECFL:  0.991505364450835
=====


In [ ]:
mid_nuscenes_5_output = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_mid.pt')
mid_nuscenes_5_gt = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_gt_futures.pt')
mid_nuscenes_5_cum_ade, mid_nuscenes_5_cum_fde = DE_by_timestep(mid_nuscenes_5_output, mid_nuscenes_5_gt)
mid_nuscenes_5_kde_nll = kde_nll(mid_nuscenes_5_output, mid_nuscenes_5_gt)
mid_nuscenes_5_ECFL = nuscenes_ECFL(solver, mid_nuscenes_5_output)
print("=====")
print("Mid Nuscenes 5")
print("ADE: ", mid_nuscenes_5_cum_ade[-1])
print("FDE: ", mid_nuscenes_5_cum_fde[-1])
print("KDE NLL: ", mid_nuscenes_5_kde_nll)
print("ECFL: ", mid_nuscenes_5_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [04:10<00:00, 13.81it/s]

=====
Mid Nuscenes 5
ADE:  2.3807406425476074
FDE:  5.540346145629883
KDE NLL:  9.336190334753748
ECFL:  0.6923791615971685
=====


In [ ]:
mid_nuscenes_5_map = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_5_mid_map.pt')
mid_nuscenes_5_map_gt = torch.load('Model_output_for_paper/nuscenes/5/nuscenes_gt_futures.pt')
mid_nuscenes_5_map_cum_ade, mid_nuscenes_5_map_cum_fde = DE_by_timestep(mid_nuscenes_5_map, mid_nuscenes_5_map_gt)
mid_nuscenes_5_map_kde_nll = kde_nll(mid_nuscenes_5_map, mid_nuscenes_5_map_gt)
mid_nuscenes_5_map_ECFL = nuscenes_ECFL(solver, mid_nuscenes_5_map)
print("=====")
print("Mid Nuscenes 5 Map")
print("ADE: ", mid_nuscenes_5_map_cum_ade[-1])
print("FDE: ", mid_nuscenes_5_map_cum_fde[-1])
print("KDE NLL: ", mid_nuscenes_5_map_kde_nll)
print("ECFL: ", mid_nuscenes_5_map_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [04:12<00:00, 13.68it/s]

=====
Mid Nuscenes 5 Map
ADE:  2.4206960201263428
FDE:  5.609602928161621
KDE NLL:  9.516958325853688
ECFL:  0.687269107399624
=====


In [18]:
# Trajectron++ Nuscenes 10
solver = eval_jux_solver(args)
trajectron_nuscenes_10 = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_trajpp.pt')
trajectron_nuscenes_10_gt = trajectron_nuscenes_10['gt']
trajectron_nuscenes_10_output = trajectron_nuscenes_10['sample']
trajectron_nuscenes_10_cum_ade, trajectron_nuscenes_10_cum_fde = DE_by_timestep(trajectron_nuscenes_10_output, trajectron_nuscenes_10_gt)
trajectron_nuscenes_10_kde_nll = kde_nll(trajectron_nuscenes_10_output, trajectron_nuscenes_10_gt)
trajectron_nuscenes_10_ECFL = nuscenes_ECFL(solver, trajectron_nuscenes_10_output)
print("=====")
print("Trajectron++ Nuscenes 10")
print("ADE: ", trajectron_nuscenes_10_cum_ade[-1])
print("FDE: ", trajectron_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", trajectron_nuscenes_10_kde_nll)
print("ECFL: ", trajectron_nuscenes_10_ECFL)
print("=====")


-------------------------- loading test data --------------------------


/tmp/ipykernel_3590186/3324040044.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trajectron_nuscenes_10 = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_tra

138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [03:44<00:00, 15.39it/s]

=====
Trajectron++ Nuscenes 10
ADE:  1.9217180013656616
FDE:  4.009583950042725
KDE NLL:  8.200012418715337
ECFL:  0.8124654352394647
=====


In [19]:
# Agentformer Nuscenes 10
solver = eval_jux_solver(args)
agentformer_nuscenes_10 = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_aformer.pt')
agentformer_nuscenes_10_gt = agentformer_nuscenes_10['gt']
agentformer_nuscenes_10_output = agentformer_nuscenes_10['sample']
agentformer_nuscenes_10_cum_ade, agentformer_nuscenes_10_cum_fde = DE_by_timestep(agentformer_nuscenes_10_output, agentformer_nuscenes_10_gt)
agentformer_nuscenes_10_kde_nll = kde_nll(agentformer_nuscenes_10_output, agentformer_nuscenes_10_gt)
agentformer_nuscenes_10_ECFL = nuscenes_ECFL(solver, agentformer_nuscenes_10_output)
print("=====")
print("Agentformer Nuscenes 10")
print("ADE: ", agentformer_nuscenes_10_cum_ade[-1])
print("FDE: ", agentformer_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", agentformer_nuscenes_10_kde_nll)
print("ECFL: ", agentformer_nuscenes_10_ECFL)
print("=====")



-------------------------- loading test data --------------------------


/tmp/ipykernel_3590186/3602562109.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  agentformer_nuscenes_10 = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_af

138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [03:43<00:00, 15.45it/s]

=====
Agentformer Nuscenes 10
ADE:  1.4518556594848633
FDE:  2.855694532394409
KDE NLL:  5.673750988343136
ECFL:  0.8426169671496516
=====


In [ ]:
# Ynet Nuscenes 10
solver = eval_jux_solver(args)
ynet_nuscenes_10 = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_ynet.pt')
ynet_nuscenes_10_gt = ynet_nuscenes_10['gt']
ynet_nuscenes_10_output = ynet_nuscenes_10['sample']
ynet_nuscenes_10_cum_ade, ynet_nuscenes_10_cum_fde = DE_by_timestep(ynet_nuscenes_10_output, ynet_nuscenes_10_gt)
ynet_nuscenes_10_kde_nll = kde_nll(ynet_nuscenes_10_output, ynet_nuscenes_10_gt)
ynet_nuscenes_10_ECFL = nuscenes_ECFL(solver, ynet_nuscenes_10_output)
print("=====")
print("Ynet Nuscenes 10")
print("ADE: ", ynet_nuscenes_10_cum_ade[-1])
print("FDE: ", ynet_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", ynet_nuscenes_10_kde_nll)
print("ECFL: ", ynet_nuscenes_10_ECFL)
print("=====")


-------------------------- loading test data --------------------------
138 sequence files are loaded ...
total num samples: 3458
------------------------------ done --------------------------------

[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|█████████▉| 3457/3458 [03:57<00:00, 14.53it/s]

=====
Ynet Nuscenes 10
ADE:  1.325176477432251
FDE:  2.0495333671569824
KDE NLL:  5.598410217136496
ECFL:  0.707134166574494
=====


In [ ]:
# Muse Nuscenes 10
muse_nuscenes_10_output = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_muse.pt')
muse_nuscenes_10_gt = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_gt_futures.pt')
muse_nuscenes_10_cum_ade, muse_nuscenes_10_cum_fde = DE_by_timestep(muse_nuscenes_10_output, muse_nuscenes_10_gt)
muse_nuscenes_10_kde_nll = kde_nll(muse_nuscenes_10_output, muse_nuscenes_10_gt)
muse_nuscenes_10_ECFL = nuscenes_ECFL(solver, muse_nuscenes_10_output)
print("=====")
print("Muse Nuscenes 10")
print("ADE: ", muse_nuscenes_10_cum_ade[-1])
print("FDE: ", muse_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", muse_nuscenes_10_kde_nll)
print("ECFL: ", muse_nuscenes_10_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [02:23<00:00, 24.05it/s]

=====
Muse Nuscenes 10
ADE:  1.1022366285324097
FDE:  2.1060445308685303
KDE NLL:  4.608262803156328
ECFL:  0.8926335582347086
=====


In [ ]:
# Diffuse Nuscenes 10
diffuse_nuscenes_10_output = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_diffuse.pt')
diffuse_nuscenes_10_gt = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_gt_futures.pt')
diffuse_nuscenes_10_cum_ade, diffuse_nuscenes_10_cum_fde = DE_by_timestep(diffuse_nuscenes_10_output, diffuse_nuscenes_10_gt)
diffuse_nuscenes_10_kde_nll = kde_nll(diffuse_nuscenes_10_output, diffuse_nuscenes_10_gt)
diffuse_nuscenes_10_ECFL = nuscenes_ECFL(solver, diffuse_nuscenes_10_output)
print("=====")
print("Diffuse Nuscenes 10")
print("ADE: ", diffuse_nuscenes_10_cum_ade[-1])
print("FDE: ", diffuse_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", diffuse_nuscenes_10_kde_nll)
print("ECFL: ", diffuse_nuscenes_10_ECFL)
print("=====")

100%|█████████▉| 3457/3458 [02:24<00:00, 23.97it/s]

=====
Diffuse Nuscenes 10
ADE:  1.4194294214248657
FDE:  2.043210506439209
KDE NLL:  5.395097205687225
ECFL:  0.9207277955978321
=====


In [ ]:
# Guided Nuscenes 10
guided_nuscenes_10_output = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_guided.pt')
guided_nuscenes_10_gt = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_gt_futures.pt')
guided_nuscenes_10_cum_ade, guided_nuscenes_10_cum_fde = DE_by_timestep(guided_nuscenes_10_output, guided_nuscenes_10_gt)
guided_nuscenes_10_kde_nll = kde_nll(guided_nuscenes_10_output, guided_nuscenes_10_gt)
guided_nuscenes_10_ECFL = nuscenes_ECFL(solver, guided_nuscenes_10_output)
print("=====")
print("Guided Nuscenes 10")
print("ADE: ", guided_nuscenes_10_cum_ade[-1])
print("FDE: ", guided_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", guided_nuscenes_10_kde_nll)
print("ECFL: ", guided_nuscenes_10_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [02:23<00:00, 24.07it/s]

=====
Guided Nuscenes 10
ADE:  1.4092073440551758
FDE:  2.0165162086486816
KDE NLL:  5.325684368161499
ECFL:  0.990819599601814
=====


In [ ]:
# MID Nuscenes 10
mid_nuscenes_10_output = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_mid.pt')
mid_nuscenes_10_gt = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_gt_futures.pt')
mid_nuscenes_10_cum_ade, mid_nuscenes_10_cum_fde = DE_by_timestep(mid_nuscenes_10_output, mid_nuscenes_10_gt)
mid_nuscenes_10_kde_nll = kde_nll(mid_nuscenes_10_output, mid_nuscenes_10_gt)
mid_nuscenes_10_ECFL = nuscenes_ECFL(solver, mid_nuscenes_10_output)
print("=====")
print("Mid Nuscenes 10")
print("ADE: ", mid_nuscenes_10_cum_ade[-1])
print("FDE: ", mid_nuscenes_10_cum_fde[-1])
print("KDE NLL: ", mid_nuscenes_10_kde_nll)
print("ECFL: ", mid_nuscenes_10_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [02:32<00:00, 22.60it/s]

=====
Mid Nuscenes 10
ADE:  1.9290512800216675
FDE:  4.2896728515625
KDE NLL:  7.4156804924620445
ECFL:  0.6897245879880545
=====


In [ ]:
# MID Nuscenes 10 Map
mid_nuscenes_10_map = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_10_mid_map.pt')
mid_nuscenes_10_map_gt = torch.load('Model_output_for_paper/nuscenes/10/nuscenes_gt_futures.pt')
mid_nuscenes_10_map_cum_ade, mid_nuscenes_10_map_cum_fde = DE_by_timestep(mid_nuscenes_10_map, mid_nuscenes_10_map_gt)
mid_nuscenes_10_map_kde_nll = kde_nll(mid_nuscenes_10_map, mid_nuscenes_10_map_gt)
mid_nuscenes_10_map_ECFL = nuscenes_ECFL(solver, mid_nuscenes_10_map)
print("=====")
print("Mid Nuscenes 10 Map")
print("ADE: ", mid_nuscenes_10_map_cum_ade[-1])
print("FDE: ", mid_nuscenes_10_map_cum_fde[-1])
print("KDE NLL: ", mid_nuscenes_10_map_kde_nll)
print("ECFL: ", mid_nuscenes_10_map_ECFL)
print("=====")


100%|█████████▉| 3457/3458 [02:32<00:00, 22.66it/s]

=====
Mid Nuscenes 10 Map
ADE:  1.9570403099060059
FDE:  4.28259801864624
KDE NLL:  7.412126992982738
ECFL:  0.6840172547284592
=====
